In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Python is a popular programming language known for its simplicity and versatility.",
        metadata={"source": "python-doc"}
    ),

    Document(
        page_content="Machine learning allows computers to learn patterns from data and make predictions.",
        metadata={"source": "ml-doc"}
    ),

    Document(
        page_content="Cloud computing provides on-demand access to computing resources over the internet.",
        metadata={"source": "cloud-doc"}
    ),

    Document(
        page_content="Docker is a platform used to package applications and their dependencies into containers.",
        metadata={"source": "docker-doc"}
    ),

    Document(
        page_content="SQL is used to store, retrieve, and manage data in relational databases.",
        metadata={"source": "sql-doc"}
    )
]

In [4]:
documents

[Document(metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.'),
 Document(metadata={'source': 'ml-doc'}, page_content='Machine learning allows computers to learn patterns from data and make predictions.'),
 Document(metadata={'source': 'cloud-doc'}, page_content='Cloud computing provides on-demand access to computing resources over the internet.'),
 Document(metadata={'source': 'docker-doc'}, page_content='Docker is a platform used to package applications and their dependencies into containers.'),
 Document(metadata={'source': 'sql-doc'}, page_content='SQL is used to store, retrieve, and manage data in relational databases.')]

In [7]:
from langchain_groq import ChatGroq

model = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x12fd9af60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12fb1db20>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

#HuggingFace Embeddings
embedding = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9617.30it/s]


In [9]:
# Vector Stores
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embedding)
vectorstore

In [10]:
vectorstore.similarity_search("python")

[Document(id='5496e768-f26b-4eda-b615-9dc7967bc346', metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.'),
 Document(id='a4964d89-538c-45bd-8048-ee71275f98cf', metadata={'source': 'ml-doc'}, page_content='Machine learning allows computers to learn patterns from data and make predictions.'),
 Document(id='f4fba653-8fd4-4761-aa6e-a08a6816b071', metadata={'source': 'docker-doc'}, page_content='Docker is a platform used to package applications and their dependencies into containers.'),
 Document(id='ff0a01f2-1f54-4e02-b8da-ff618ae73b32', metadata={'source': 'sql-doc'}, page_content='SQL is used to store, retrieve, and manage data in relational databases.')]

In [11]:
# Async Query 
await vectorstore.asimilarity_search("python")

[Document(id='5496e768-f26b-4eda-b615-9dc7967bc346', metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.'),
 Document(id='a4964d89-538c-45bd-8048-ee71275f98cf', metadata={'source': 'ml-doc'}, page_content='Machine learning allows computers to learn patterns from data and make predictions.'),
 Document(id='f4fba653-8fd4-4761-aa6e-a08a6816b071', metadata={'source': 'docker-doc'}, page_content='Docker is a platform used to package applications and their dependencies into containers.'),
 Document(id='ff0a01f2-1f54-4e02-b8da-ff618ae73b32', metadata={'source': 'sql-doc'}, page_content='SQL is used to store, retrieve, and manage data in relational databases.')]

In [12]:
vectorstore.similarity_search_with_score("python")

[(Document(id='5496e768-f26b-4eda-b615-9dc7967bc346', metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.'),
  0.3167514503002167),
 (Document(id='a4964d89-538c-45bd-8048-ee71275f98cf', metadata={'source': 'ml-doc'}, page_content='Machine learning allows computers to learn patterns from data and make predictions.'),
  0.6653919219970703),
 (Document(id='f4fba653-8fd4-4761-aa6e-a08a6816b071', metadata={'source': 'docker-doc'}, page_content='Docker is a platform used to package applications and their dependencies into containers.'),
  0.7894720435142517),
 (Document(id='ff0a01f2-1f54-4e02-b8da-ff618ae73b32', metadata={'source': 'sql-doc'}, page_content='SQL is used to store, retrieve, and manage data in relational databases.'),
  0.8406028747558594)]

## Retrievers

In [ ]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["python", "docker"])

[[Document(id='5496e768-f26b-4eda-b615-9dc7967bc346', metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.')],
 [Document(id='f4fba653-8fd4-4761-aa6e-a08a6816b071', metadata={'source': 'docker-doc'}, page_content='Docker is a platform used to package applications and their dependencies into containers.')]]

In [14]:
vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

retriever.batch(["python", "sql"])

[[Document(id='5496e768-f26b-4eda-b615-9dc7967bc346', metadata={'source': 'python-doc'}, page_content='Python is a popular programming language known for its simplicity and versatility.')],
 [Document(id='ff0a01f2-1f54-4e02-b8da-ff618ae73b32', metadata={'source': 'sql-doc'}, page_content='SQL is used to store, retrieve, and manage data in relational databases.')]]

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context":retriever, "question":RunnablePassthrough()}|prompt|model

response = rag_chain.invoke("Tell me about docker")
response.content

'Docker is a platform used to package applications and their dependencies into containers.'